# 02 – Feature Engineering & Data Preparation

Neste notebook vamos:
1. Tratar valores ausentes (`msgs_tot`).
2. Reduzir assimetria aplicando `np.log1p` nas variáveis com **skew > 3**.
3. Criar features derivadas identificadas na EDA (ex.: `wifi_mobile_ratio`).
4. Padronizar escalas numéricas com `StandardScaler`.
5. Salvar:  
   • dataset final (`/data/processed/train_ready.parquet`)  
   • pipeline de pré-processamento (`/outputs/pipelines/preprocess.joblib`)

#### 1 ▸ Imports

In [10]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from utils.data_prep import create_label

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#### 2 ▸ Carregar dados

In [12]:
DATA_RAW = Path("../data/Moto_HW_DATA_hashed.csv")
df = pd.read_csv(DATA_RAW)
df = create_label(df)

# Mesmas colunas estudadas na EDA
cols = [
    "full_cap_min", "btlow_total", "dchrg_diff", "total_chrg_cnt",
    "son_hrs", "msgs_tot", "wifi_hrs", "mobile_rx_MB"
]
df = df[cols + ["label_ineficiente"]]  

#### 3 ▸ Missing values

In [13]:
# msgs_tot é a única coluna com NaN – preenchemos com 0 (assume “sem mensagens”)
df["msgs_tot"].fillna(0, inplace=True)

/var/folders/nk/hhm4r_k11vj25bs27s2w2ch40000gn/T/ipykernel_85423/2005398948.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["msgs_tot"].fillna(0, inplace=True)


#### 4 ▸ Transformações log1p

In [14]:
skewed_cols = [
    "btlow_total", "wifi_hrs", "msgs_tot",
    "total_chrg_cnt", "mobile_rx_MB", "full_cap_min"
]

for col in skewed_cols:
    df[f"{col}_log"] = np.log1p(df[col])

/Users/almthiessen/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


#### 5 ▸ Features derivadas

In [19]:
# Razão entre uso Wi-Fi e dados móveis
df["wifi_mobile_ratio"] = df["wifi_hrs"] / (df["mobile_rx_MB"] + 1)
df["wifi_mobile_ratio"] = df["wifi_hrs"] / df["mobile_rx_MB"].replace(0, np.nan)
df["wifi_mobile_ratio"] = df["wifi_mobile_ratio"].fillna(0).clip(upper=50)

# “Drain rate” aproximado (útil para análise futura)
df["drain_rate"] = (100 - df["dchrg_diff"].clip(upper=99)) / (df["son_hrs"] + 0.5)

#### 6 ▸ Selecionar X, Y

In [16]:
feature_cols = [
    # versões log
    "btlow_total_log", "wifi_hrs_log", "msgs_tot_log",
    "total_chrg_cnt_log", "mobile_rx_MB_log", "full_cap_min_log",
    # originais “suaves”
    "dchrg_diff", "son_hrs",
    # derivadas
    "wifi_mobile_ratio", "drain_rate"
]

X = df[feature_cols]
y = df["label_ineficiente"]

#### 7 ▸ Train / test split

In [17]:
# Stratify para manter proporção do rótulo; seed fixa p/ reprodutibilidade
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Train shape: {X_train.shape}  |  Test shape: {X_test.shape}")

Train shape: (16850, 10)  |  Test shape: (4213, 10)


#### 8 ▸ Pipeline de pré-processamento

In [21]:
numeric_features = feature_cols
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ],
    remainder="drop"
)

# Substitui infinitos por NaN e depois preenche com 0
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

# Ajusta apenas nos dados de treino
preprocess.fit(X_train)

# Transforma e cria DataFrames prontos
X_train_proc = preprocess.transform(X_train)
X_test_proc  = preprocess.transform(X_test)

# Converte para DataFrame para facilitar debug/SHAP
X_train_proc = pd.DataFrame(X_train_proc, columns=numeric_features, index=X_train.index)
X_test_proc  = pd.DataFrame(X_test_proc,  columns=numeric_features, index=X_test.index)

#### 9 ▸ Exportar dados & artefatos

In [24]:
# Diretórios de saída
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../outputs/pipelines").mkdir(parents=True, exist_ok=True)

# Salva conjuntos prontos em Parquet (compacto + rápido)
X_train_proc.to_parquet("../data/processed/X_train.parquet")
X_test_proc.to_parquet ("../data/processed/X_test.parquet")
X_train_proc.to_csv("../data/processed/X_train.csv", index=False)
X_test_proc.to_csv ("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv ("../data/processed/y_test.csv",  index=False)

# Salva pipeline para reuso nos modelos (03_modeling.ipynb)
joblib.dump(preprocess, "../outputs/pipelines/preprocess.joblib")

print("✅ Dados processados e pipeline salvos com sucesso.")

✅ Dados processados e pipeline salvos com sucesso.


#### 10 ▸ Checklist & próximos passos

##### ✅ Checklist de saída
- [x] Arquivos salvos em `/data/processed/`:
  - `X_train.parquet`, `X_test.parquet`
  - `y_train.csv`, `y_test.csv`
- [x] Pipeline `preprocess.joblib` em `/outputs/pipelines/`

##### Próximos passos
1. **03_modeling.ipynb**  
   - Carregar `X_train`/`y_train`.
   - Comparar modelos:  
     • DummyClassifier (baseline)  
     • LogisticRegression (com `class_weight='balanced'`)  
     • LightGBM (tree-based)
2. Validar grupo-k-fold se adicionarmos `barcode_hashed`.
3. Gerar `metrics.json` e gráficos ROC/PR para o notebook de explainability.